# LazyRegressor screening + top-3 tuning — bus dataset

Single bus (chosen in the dataset build — see `datasets/build_report.md`). Row/feature counts
and the train/test split are printed in L1 below, never hard-coded here (v2 rebuilt the dataset
with a whole-service-day split, so old counts are stale).
Target: `delay_s` (seconds late vs TransLoc ETA). Follows `plan/04_models/01_protocol.md` + `02_lazy.md`:

1. **Screen** a curated set of fast/medium sklearn (+ LightGBM) regressors with `LazyRegressor` on a
   subsampled, group-disjoint inner train/val split of TRAIN.
2. **Pick the top 3** by validation R².
3. **Tune** each of the 3 with Optuna (`GroupKFold(3)`, objective = mean MAE) on a train subsample.
4. **Refit** each tuned model on the FULL train, predict TEST **once**, save via `mc_common.save_result`.

The test split is touched exactly once per model, at the very end.

## L1 — Setup

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

import numpy as np
import pandas as pd

from mc_common import SMOKE, THREADS, budget, set_seed, load_model_ready, group_val_split, metrics, save_result

set_seed(42)
pd.set_option("display.max_columns", 100)

DS = "bus"
d = load_model_ready(DS)

print("SMOKE:", SMOKE, " THREADS:", THREADS)
print("features:", len(d.features), " target:", d.target)
print("train:", d.X_train.shape, " test:", d.X_test.shape)
print("train groups (unique cv_group):", len(np.unique(d.g_train)))


SMOKE: False  THREADS: 2
features: 35  target: delay_s
train: (16456, 35)  test: (7536, 35)
train groups (unique cv_group): 5


## L2 — Inner train/val split (for screening only)

`group_val_split` makes a group-disjoint 80/20 split of TRAIN (never touches TEST). For the
screening step we further subsample both pieces so ~24 candidate models fit quickly:
inner-train to `budget(15000, 2000)` rows, inner-val to `budget(6000, 1000)` rows.

In [2]:
X_tr, X_val, y_tr, y_val, g_tr = group_val_split(d.X_train, d.y_train, d.g_train, frac=0.2, seed=42)

n_tr = min(len(X_tr), budget(15000, 2000))
n_val = min(len(X_val), budget(6000, 1000))
idx_tr = X_tr.sample(n=n_tr, random_state=42).index
idx_val = X_val.sample(n=n_val, random_state=42).index

X_tr_s, y_tr_s = X_tr.loc[idx_tr], y_tr.loc[idx_tr]
X_val_s, y_val_s = X_val.loc[idx_val], y_val.loc[idx_val]

print("screening inner-train:", X_tr_s.shape, " inner-val:", X_val_s.shape)


screening inner-train: (14164, 35)  inner-val: (2292, 35)


## L3 — LazyRegressor screening

`CURATED` regressors (fast-to-medium, all handle a possibly-negative target):
LinearRegression, Ridge, Lasso, ElasticNet, Lars, LassoLars, OrthogonalMatchingPursuit,
BayesianRidge, HuberRegressor, SGDRegressor, PassiveAggressiveRegressor, LinearSVR,
KNeighborsRegressor, DecisionTreeRegressor, ExtraTreeRegressor, RandomForestRegressor,
ExtraTreesRegressor, BaggingRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor,
AdaBoostRegressor, MLPRegressor, LGBMRegressor, DummyRegressor (baseline, excluded from top-3).

**Excluded, and why:**
- `XGBRegressor` — has its own notebook (`xgboost_bus.ipynb`).
- `SVR` / `NuSVR` / `KernelRidge` / `GaussianProcessRegressor` — O(n²)–O(n³) in n_samples, too
  slow to screen.
- `QuantileRegressor` / `TheilSenRegressor` / `RANSACRegressor` — slow or unstable on this size
  of data.
- `PoissonRegressor` / `GammaRegressor` / `TweedieRegressor` — need a strictly positive target;
  `delay_s` can be negative (bus arriving early).

In [3]:
from lazypredict.Supervised import LazyRegressor
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, Lars, LassoLars,
    OrthogonalMatchingPursuit, BayesianRidge, HuberRegressor, SGDRegressor,
    PassiveAggressiveRegressor,
)
from sklearn.svm import LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor,
    GradientBoostingRegressor, HistGradientBoostingRegressor, AdaBoostRegressor,
)
from sklearn.neural_network import MLPRegressor
from sklearn.dummy import DummyRegressor
from lightgbm import LGBMRegressor

CURATED = [
    LinearRegression, Ridge, Lasso, ElasticNet, Lars, LassoLars,
    OrthogonalMatchingPursuit, BayesianRidge, HuberRegressor, SGDRegressor,
    PassiveAggressiveRegressor, LinearSVR, KNeighborsRegressor,
    DecisionTreeRegressor, ExtraTreeRegressor, RandomForestRegressor,
    ExtraTreesRegressor, BaggingRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, AdaBoostRegressor, MLPRegressor,
    LGBMRegressor, DummyRegressor,
]
MODEL_CLASSES = {c.__name__: c for c in CURATED}

reg = LazyRegressor(verbose=0, ignore_warnings=True, random_state=42, regressors=CURATED)
leaderboard, _ = reg.fit(X_tr_s, X_val_s, y_tr_s, y_val_s)
leaderboard


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
LinearSVR,6.758442e-01,6.807963e-01,1.444217e+02,0.158198
ExtraTreesRegressor,6.164018e-01,6.222621e-01,1.571064e+02,14.471259
HistGradientBoostingRegressor,5.750127e-01,5.815053e-01,1.653649e+02,3.000265
AdaBoostRegressor,5.471724e-01,5.540903e-01,1.706954e+02,3.023777
LGBMRegressor,5.306868e-01,5.378565e-01,1.737748e+02,21.208357
ElasticNet,5.084608e-01,5.159702e-01,1.778421e+02,0.077518
GradientBoostingRegressor,4.305183e-01,4.392184e-01,1.914235e+02,5.198008
Lasso,4.132541e-01,4.222179e-01,1.943034e+02,1.164631
LassoLars,3.907748e-01,4.000820e-01,1.979905e+02,0.062397


## L4 — Pick top 3 by validation R² (DummyRegressor excluded as a baseline, not a candidate)

In [4]:
ranked = leaderboard.drop(index="DummyRegressor", errors="ignore").sort_values("R-Squared", ascending=False)
top3 = list(ranked.index[:3])
print("top 3 by validation R2:", top3)
ranked.loc[top3, ["R-Squared", "RMSE", "Time Taken"]]


top 3 by validation R2: ['LinearSVR', 'ExtraTreesRegressor', 'HistGradientBoostingRegressor']


,R-Squared,RMSE,Time Taken
Model,,,
LinearSVR,0.680796,144.421677,0.158198
ExtraTreesRegressor,0.622262,157.106357,14.471259
HistGradientBoostingRegressor,0.581505,165.364949,3.000265


## L5 — Optuna tuning of the top 3

Each candidate gets its own search space (defined for **every** CURATED model, so whichever 3
screen best can be tuned without extra code). Linear / KNN / MLP / SVR models are wrapped in
`Pipeline([StandardScaler(), model])` since they're scale-sensitive; trees/ensembles are not.
`HistGradientBoostingRegressor` sets `early_stopping=False` (its default early stopping holds out
a random, non-grouped 10% of whatever it's fit on) and tunes `max_iter` instead.

Tuning data: a train subsample (≤ `budget(30000, 3000)` rows, group-aligned), scored with
`GroupKFold(3)` on `cv_group`, objective = mean MAE (seconds) across folds, minimised. Per-model
Optuna budget: `n_trials=budget(40, 3)`, `timeout=budget(900, 30)`s.

In [5]:
import inspect
import optuna
from optuna.samplers import TPESampler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

SCALE_WRAP = {
    "LinearRegression", "Ridge", "Lasso", "ElasticNet", "Lars", "LassoLars",
    "OrthogonalMatchingPursuit", "BayesianRidge", "HuberRegressor", "SGDRegressor",
    "PassiveAggressiveRegressor", "LinearSVR", "KNeighborsRegressor", "MLPRegressor",
}
N_FEAT = len(d.features)


def suggest_params(trial, name):
    """Hyperparameter space per CURATED model name. Returns a plain dict of ctor kwargs
    (random_state / n_jobs are added later, only for models that accept them)."""
    if name in ("RandomForestRegressor", "ExtraTreesRegressor"):
        depth_none = trial.suggest_categorical(f"{name}_depth_none", [True, False])
        max_depth = None if depth_none else trial.suggest_int(f"{name}_max_depth", 5, 40)
        feat_kind = trial.suggest_categorical(f"{name}_feat_kind", ["sqrt", "frac"])
        max_features = "sqrt" if feat_kind == "sqrt" else trial.suggest_float(f"{name}_max_features_frac", 0.3, 1.0)
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 100, 600),
            max_depth=max_depth,
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
            max_features=max_features,
        )
    if name == "HistGradientBoostingRegressor":
        return dict(
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            max_iter=trial.suggest_int("max_iter", 100, 1000),
            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 255),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 200),
            l2_regularization=trial.suggest_float("l2_regularization", 0.0, 10.0),
            # v2: default early stopping holds out a random, non-grouped 10% -- disable it and
            # let max_iter (tuned above) control tree count instead. Flows into both the tuning
            # objective and the L6 final refit since both call make_estimator on this same dict.
            early_stopping=False,
        )
    if name == "LGBMRegressor":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 100, 2000),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int("num_leaves", 15, 255),
            min_child_samples=trial.suggest_int("min_child_samples", 5, 200),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            subsample_freq=1,
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            verbose=-1,
        )
    if name == "GradientBoostingRegressor":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 100, 800),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 50),
        )
    if name == "BaggingRegressor":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 10, 200),
            max_samples=trial.suggest_float("max_samples", 0.3, 1.0),
            max_features=trial.suggest_float("max_features", 0.3, 1.0),
        )
    if name == "AdaBoostRegressor":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 50, 500),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 2.0, log=True),
            loss=trial.suggest_categorical("loss", ["linear", "square", "exponential"]),
        )
    if name in ("DecisionTreeRegressor", "ExtraTreeRegressor"):
        depth_none = trial.suggest_categorical("depth_none", [True, False])
        max_depth = None if depth_none else trial.suggest_int("max_depth", 3, 30)
        return dict(
            max_depth=max_depth,
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 50),
        )
    if name == "KNeighborsRegressor":
        return dict(
            n_neighbors=trial.suggest_int("n_neighbors", 3, 100),
            weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
            p=trial.suggest_categorical("p", [1, 2]),
        )
    if name == "MLPRegressor":
        hidden = trial.suggest_categorical("hidden_layer_sizes", ["64", "128_64", "256_128", "128_128_64"])
        hidden_map = {"64": (64,), "128_64": (128, 64), "256_128": (256, 128), "128_128_64": (128, 128, 64)}
        return dict(
            hidden_layer_sizes=hidden_map[hidden],
            alpha=trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
            learning_rate_init=trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
            early_stopping=True,
            max_iter=300,
        )
    if name == "LinearSVR":
        return dict(
            C=trial.suggest_float("C", 1e-3, 10.0, log=True),
            epsilon=trial.suggest_float("epsilon", 0.0, 1.0),
            max_iter=5000,
        )
    if name == "PassiveAggressiveRegressor":
        return dict(
            C=trial.suggest_float("C", 1e-3, 10.0, log=True),
            epsilon=trial.suggest_float("epsilon", 0.01, 0.5),
            max_iter=2000,
        )
    if name == "SGDRegressor":
        return dict(
            alpha=trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
            l1_ratio=trial.suggest_float("l1_ratio", 0.0, 1.0),
            penalty=trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"]),
            learning_rate=trial.suggest_categorical("sgd_learning_rate", ["invscaling", "optimal", "adaptive"]),
            eta0=trial.suggest_float("eta0", 1e-4, 1e-1, log=True),
            max_iter=2000,
        )
    if name == "HuberRegressor":
        return dict(
            epsilon=trial.suggest_float("epsilon", 1.05, 2.0),
            alpha=trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
            max_iter=500,
        )
    if name == "BayesianRidge":
        return dict(
            alpha_1=trial.suggest_float("alpha_1", 1e-8, 1e-2, log=True),
            lambda_1=trial.suggest_float("lambda_1", 1e-8, 1e-2, log=True),
        )
    if name == "Ridge":
        return dict(alpha=trial.suggest_float("alpha", 1e-3, 100.0, log=True))
    if name in ("Lasso", "LassoLars"):
        return dict(alpha=trial.suggest_float("alpha", 1e-4, 10.0, log=True))
    if name == "ElasticNet":
        return dict(
            alpha=trial.suggest_float("alpha", 1e-4, 10.0, log=True),
            l1_ratio=trial.suggest_float("l1_ratio", 0.05, 0.95),
        )
    if name in ("Lars", "OrthogonalMatchingPursuit"):
        return dict(n_nonzero_coefs=trial.suggest_int("n_nonzero_coefs", 1, N_FEAT))
    if name == "LinearRegression":
        return dict(fit_intercept=trial.suggest_categorical("fit_intercept", [True, False]))
    raise ValueError(f"no search space defined for {name}")


def make_estimator(name, params):
    """Instantiate the model, adding random_state/n_jobs only where the class supports them,
    and wrapping scale-sensitive models in a StandardScaler pipeline."""
    cls = MODEL_CLASSES[name]
    sig = inspect.signature(cls.__init__).parameters
    kwargs = dict(params)
    if "random_state" in sig:
        kwargs["random_state"] = 42
    if "n_jobs" in sig:
        kwargs["n_jobs"] = THREADS
    est = cls(**kwargs)
    if name in SCALE_WRAP:
        est = Pipeline([("scaler", StandardScaler()), ("model", est)])
    return est


# tuning data: train subsample, group-aligned
n_tune = min(len(d.X_train), budget(30000, 3000))
tune_idx = d.X_train.sample(n=n_tune, random_state=42).index
pos = d.X_train.index.get_indexer(tune_idx)
X_tune = d.X_train.loc[tune_idx].reset_index(drop=True)
y_tune = d.y_train.loc[tune_idx].reset_index(drop=True)
g_tune = d.g_train[pos]
print("tuning subsample:", X_tune.shape, " unique groups:", len(np.unique(g_tune)))


tuning subsample: (16456, 35)  unique groups: 5


In [6]:
def objective(trial, name):
    # suggest_params uses name-prefixed / conditional optuna param names for a few models
    # (e.g. RF's max_depth is only drawn when depth_none=False), so study.best_params is not
    # reliably the sklearn kwarg dict. Stash the actual dict as a user attr and read it back
    # from the winning trial instead of reconstructing it from study.best_params.
    params = suggest_params(trial, name)
    trial.set_user_attr("model_params", params)
    maes = []
    for tr_idx, va_idx in GroupKFold(n_splits=3).split(X_tune, y_tune, g_tune):
        est = make_estimator(name, params)
        est.fit(X_tune.iloc[tr_idx], y_tune.iloc[tr_idx])
        pred = est.predict(X_tune.iloc[va_idx])
        maes.append(mean_absolute_error(y_tune.iloc[va_idx], pred))
    return float(np.mean(maes))


tuned = {}  # name -> dict(best_params, best_mae, study, n_trials)
for name in top3:
    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
    study.optimize(lambda t: objective(t, name), n_trials=budget(40, 3), timeout=budget(900, 30))
    best_params = study.best_trial.user_attrs["model_params"]
    n_trials_complete = len(study.trials)
    tuned[name] = {"best_params": best_params, "best_mae": study.best_value,
                   "study": study, "n_trials": n_trials_complete}
    print(f"{name}: best CV MAE={study.best_value:.1f}s  n_trials={n_trials_complete}  params={best_params}")


LinearSVR: best CV MAE=286.9s  n_trials=40  params={'C': 0.018528293586170495, 'epsilon': 0.09348960705931308, 'max_iter': 5000}


ExtraTreesRegressor: best CV MAE=168.5s  n_trials=40  params={'n_estimators': 202, 'max_depth': 22, 'min_samples_leaf': 3, 'max_features': 'sqrt'}


HistGradientBoostingRegressor: best CV MAE=214.2s  n_trials=40  params={'learning_rate': 0.01196570826306683, 'max_iter': 200, 'max_leaf_nodes': 204, 'min_samples_leaf': 200, 'l2_regularization': 8.731155660238667, 'early_stopping': False}


## L6 — Final fit on FULL train, predict TEST once

Each tuned model is refit on the entire train split (no subsampling) with its best params, then
scored on TEST exactly once via `mc_common.save_result` (writes the result JSON + predictions csv;
family becomes `lazy_smoke` under `MC_SMOKE=1`).

In [7]:
results_rows = []
for rank, name in enumerate(top3, start=1):
    info = tuned[name]
    t0 = time.time()
    est = make_estimator(name, info["best_params"])
    est.fit(d.X_train, d.y_train)
    y_pred = est.predict(d.X_test)
    fit_s = time.time() - t0

    leaderboard_top10 = ranked.head(10).reset_index()[["Model", "R-Squared", "RMSE"]].to_dict(orient="records")
    rec = save_result(
        DS, "lazy", name, d.y_test, y_pred, info["best_params"],
        cv_mae=info["best_mae"],
        extra={"screen_rank": rank, "screen_val_r2": float(ranked.loc[name, "R-Squared"]),
               "leaderboard_top10": leaderboard_top10, "fit_seconds": fit_s,
               "n_trials_complete": info["n_trials"]},
    )
    results_rows.append({"model": name, "fit_seconds": fit_s, **{k: rec[k] for k in ("r2", "mae", "rmse")}})


[bus__lazy__LinearSVR] test R2=0.3320  MAE=141.6s  RMSE=189.8s  (n=7536)


[bus__lazy__ExtraTreesRegressor] test R2=0.3865  MAE=138.7s  RMSE=181.9s  (n=7536)


[bus__lazy__HistGradientBoostingRegressor] test R2=0.5043  MAE=127.4s  RMSE=163.5s  (n=7536)


## L7 — Summary

In [8]:
summary = pd.DataFrame(results_rows).set_index("model")
print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
summary


                               fit_seconds    r2     mae    rmse
model                                                           
LinearSVR                            0.051 0.332 141.573 189.773
ExtraTreesRegressor                  1.299 0.386 138.691 181.877
HistGradientBoostingRegressor        1.225 0.504 127.380 163.485


,fit_seconds,r2,mae,rmse
model,,,,
LinearSVR,0.051005,0.332029,141.572527,189.773361
ExtraTreesRegressor,1.299094,0.386460,138.690544,181.876953
HistGradientBoostingRegressor,1.225255,0.504272,127.379677,163.485087
